### This notebook trains and evaluates a machine learning model using Gold-layer features to predict COVID risk levels.

In this step, AI models are built on top of feature-engineered Gold data, with experiments tracked using MLflow to generate interpretable and decision-ready insights.


In [0]:
covid_gold_df = spark.read.table("workspace.default.covid_gold")


In [0]:
covid_gold_df.printSchema()


root
 |-- date: date (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- cum_cases: integer (nullable = true)
 |-- new_death: integer (nullable = true)
 |-- cum_death: integer (nullable = true)
 |-- new_recovered: integer (nullable = true)
 |-- cum_recovered: integer (nullable = true)
 |-- cum_active_cases: integer (nullable = true)
 |-- previous_day_cases: integer (nullable = true)
 |-- daily_case_growth_rate: double (nullable = true)
 |-- active_case_ratio: double (nullable = true)
 |-- mortality_rate: double (nullable = true)



###Risk Labels
COVID risk levels are derived using domain-informed thresholds on growth, burden, and severity indicators.

In [0]:
from pyspark.sql.functions import when, col

covid_labeled_df = (
    covid_gold_df
    .withColumn(
        "risk_level",
        when(
            (col("daily_case_growth_rate") > 0.1) |
            (col("active_case_ratio") > 0.7) |
            (col("mortality_rate") > 0.02),
            "High"
        )
        .when(
            (col("daily_case_growth_rate") > 0.02) |
            (col("active_case_ratio") > 0.3),
            "Medium"
        )
        .otherwise("Low")
    )
)


### Verify risk labels

In [0]:
display(
    covid_labeled_df.select(
        "date",
        "daily_case_growth_rate",
        "active_case_ratio",
        "mortality_rate",
        "risk_level"
    ).limit(15)
)


date,daily_case_growth_rate,active_case_ratio,mortality_rate,risk_level
2020-01-30,null,1.0,0.0,High
2020-01-31,-1.0,1.0,0.0,High
2020-02-01,null,1.0,0.0,High
2020-02-02,null,1.0,0.0,High
2020-02-03,0.0,1.0,0.0,High
2020-02-04,-1.0,1.0,0.0,High
2020-02-05,null,1.0,0.0,High
2020-02-06,null,1.0,0.0,High
2020-02-07,null,1.0,0.0,High
2020-02-08,null,1.0,0.0,High


### Why daily_case_growth_rate is null?
Daily case growth rate is naturally undefined for initial dates and zero-case transitions in time-series data.


### Prepare Data for ML :
Null values are handled and relevant Gold features are selected to prepare a clean, model-ready dataset.


In [0]:
#handle NULLs
ml_ready_df = (
    covid_labeled_df
    .fillna({
        "daily_case_growth_rate": 0
    })
)


In [0]:
final_ml_df = ml_ready_df.select(
    "date",
    "daily_case_growth_rate",
    "active_case_ratio",
    "mortality_rate",
    "risk_level"
)


In [0]:
display(final_ml_df.limit(10))


date,daily_case_growth_rate,active_case_ratio,mortality_rate,risk_level
2020-01-30,0.0,1.0,0.0,High
2020-01-31,-1.0,1.0,0.0,High
2020-02-01,0.0,1.0,0.0,High
2020-02-02,0.0,1.0,0.0,High
2020-02-03,0.0,1.0,0.0,High
2020-02-04,-1.0,1.0,0.0,High
2020-02-05,0.0,1.0,0.0,High
2020-02-06,0.0,1.0,0.0,High
2020-02-07,0.0,1.0,0.0,High
2020-02-08,0.0,1.0,0.0,High


###Encode Labels & Assemble Features

In [0]:
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="risk_level",
    outputCol="risk_label"
)

indexed_df = label_indexer.fit(final_ml_df).transform(final_ml_df)


In [0]:
from pyspark.ml.feature import VectorAssembler

feature_assembler = VectorAssembler(
    inputCols=[
        "daily_case_growth_rate",
        "active_case_ratio",
        "mortality_rate"
    ],
    outputCol="features"
)

ml_dataset = feature_assembler.transform(indexed_df)


In [0]:
#Verify ML-ready dataset
display(
    ml_dataset.select(
        "daily_case_growth_rate",
        "active_case_ratio",
        "mortality_rate",
        "risk_level",
        "risk_label",
        "features"
    ).limit(10)
)


daily_case_growth_rate,active_case_ratio,mortality_rate,risk_level,risk_label,features
0.0,1.0,0.0,High,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0"",""0.0""]}"
-1.0,1.0,0.0,High,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.0"",""1.0"",""0.0""]}"
0.0,1.0,0.0,High,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0"",""0.0""]}"
0.0,1.0,0.0,High,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0"",""0.0""]}"
0.0,1.0,0.0,High,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0"",""0.0""]}"
-1.0,1.0,0.0,High,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.0"",""1.0"",""0.0""]}"
0.0,1.0,0.0,High,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0"",""0.0""]}"
0.0,1.0,0.0,High,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0"",""0.0""]}"
0.0,1.0,0.0,High,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0"",""0.0""]}"
0.0,1.0,0.0,High,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0"",""0.0""]}"


###Train ML Model & Track with MLflow

In [0]:
#Split data into Train & Test
train_df, test_df = ml_dataset.randomSplit([0.8, 0.2], seed=42)


In [0]:
import mlflow
import mlflow.spark


In [0]:
mlflow.set_experiment("/Users/ruchiwange@gmail.com/COVID_Risk_Prediction")

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/1711285042330077', creation_time=1769716196375, experiment_id='1711285042330077', last_update_time=1769716563045, lifecycle_stage='active', name='/Users/ruchiwange@gmail.com/COVID_Risk_Prediction', tags={'mlflow.experiment.sourceName': '/Users/ruchiwange@gmail.com/COVID_Risk_Prediction',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'ruchiwange@gmail.com',
 'mlflow.ownerId': '76361278390427'}>

In [0]:
import os

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/raw_data"


In [0]:
#Train Logistic Regression model
from pyspark.ml.classification import LogisticRegression

with mlflow.start_run():
    
    lr = LogisticRegression(
        featuresCol="features",
        labelCol="risk_label"
    )
    
    lr_model = lr.fit(train_df)
    
    predictions = lr_model.transform(test_df)
    
    # Log model
    mlflow.spark.log_model(lr_model, "logistic_regression_model")


2026/01/29 20:58:50 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.2.2) contains a local version label (+databricks.connect.17.2.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/01/29 20:58:55 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-91dacdef-736c-481f-93b2-fb/tmpua9yks4y/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/01/29 20:58:55 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [0]:
#Evaluate model performance
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="risk_label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
accuracy


0.8235294117647058

In [0]:
#Log metric to MLflow:
mlflow.log_metric("accuracy", accuracy)


###Interpret Model Results & Generate AI Insights

model predictions:

In [0]:
display(
    predictions.select(
        "daily_case_growth_rate",
        "active_case_ratio",
        "mortality_rate",
        "risk_level",
        "prediction"
    ).limit(15)
)


daily_case_growth_rate,active_case_ratio,mortality_rate,risk_level,prediction
0.0,1.0,0.0,High,1.0
0.0,1.0,0.0,High,1.0
0.0,1.0,0.0,High,1.0
0.0,1.0,0.0,High,1.0
0.0,0.3333333333333333,0.0,Medium,2.0
0.0,0.0,0.0,Low,0.0
0.0,0.0,0.0,Low,0.0
-0.9090909090909091,0.9,0.0,High,0.0
-0.09090909090909091,0.8660714285714286,0.017857142857142856,High,1.0
0.4,0.873015873015873,0.015873015873015872,High,1.0


In [0]:
#Converting predictions into readable risk labels
from pyspark.sql.functions import when

predicted_df = predictions.select(
    "date",
    "daily_case_growth_rate",
    "active_case_ratio",
    "mortality_rate",
    "risk_level",
    "prediction"
).withColumn(
    "predicted_risk_level",
    when(col("prediction") == 0, "Low")
    .when(col("prediction") == 1, "Medium")
    .otherwise("High")
)



In [0]:
#final AI output
display(
    predicted_df.select(
        "daily_case_growth_rate",
        "active_case_ratio",
        "mortality_rate",
        "risk_level",
        "predicted_risk_level"
    ).limit(15)
)


daily_case_growth_rate,active_case_ratio,mortality_rate,risk_level,predicted_risk_level
0.0,1.0,0.0,High,Medium
0.0,1.0,0.0,High,Medium
0.0,1.0,0.0,High,Medium
0.0,1.0,0.0,High,Medium
0.0,0.3333333333333333,0.0,Medium,High
0.0,0.0,0.0,Low,Low
0.0,0.0,0.0,Low,Low
-0.9090909090909091,0.9,0.0,High,Low
-0.09090909090909091,0.8660714285714286,0.017857142857142856,High,Medium
0.4,0.873015873015873,0.015873015873015872,High,Medium


###AI Insights Generation


In [0]:
predicted_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.covid_ai_insights")


In [0]:
spark.sql("DESCRIBE TABLE workspace.default.covid_ai_insights").show()


+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|daily_case_growth...|   double|   NULL|
|   active_case_ratio|   double|   NULL|
|      mortality_rate|   double|   NULL|
|          risk_level|   string|   NULL|
|          risk_label|   double|   NULL|
|            features|   vector|   NULL|
|       rawPrediction|   vector|   NULL|
|         probability|   vector|   NULL|
|          prediction|   double|   NULL|
|predicted_risk_level|   string|   NULL|
|                date|     date|   NULL|
+--------------------+---------+-------+



In [0]:
spark.sql("SHOW TABLES IN workspace.default").show()


+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
| default| covid_ai_insights|      false|
| default|        covid_gold|      false|
| default|      covid_silver|      false|
| default|vaccination_silver|      false|
+--------+------------------+-----------+



In [0]:
spark.sql("SELECT * FROM workspace.default.covid_ai_insights LIMIT 5").show()


+----------------------+--------------------+--------------------+----------+----------+--------------------+--------------------+--------------------+----------+--------------------+
|daily_case_growth_rate|   active_case_ratio|      mortality_rate|risk_level|risk_label|            features|       rawPrediction|         probability|prediction|predicted_risk_level|
+----------------------+--------------------+--------------------+----------+----------+--------------------+--------------------+--------------------+----------+--------------------+
|   -0.9090909090909091|                 0.9|                 0.0|      High|       1.0|[-0.9090909090909...|[6.59595140544543...|[0.99871167541827...|       0.0|                 Low|
|    -0.447868439571836| 0.00883324780599372|0.013261417157842906|       Low|       0.0|[-0.4478684395718...|[9.85962730486856...|[0.99998545612208...|       0.0|                 Low|
|  -0.31242442563482464|0.016810180773080635|0.014332024436732418|       Low|   

In [0]:
spark.sql(
    "SELECT date, predicted_risk_level FROM workspace.default.covid_ai_insights LIMIT 5"
).show()


+----------+--------------------+
|      date|predicted_risk_level|
+----------+--------------------+
|2020-02-01|              Medium|
|2020-02-05|              Medium|
|2020-02-07|              Medium|
|2020-02-12|              Medium|
|2020-02-18|                High|
+----------+--------------------+

